Sizing the 5-transistor OTA example with Evolutionary Optimization (Nevergrad) as a constraint satisfaction problem.

# Pre-body

## Clearing past runs (optional)

In [ ]:
!rm -rf logs/ # clear logs
!rm -rf spice_out/

## IIC-OSIC Env Setup

In [ ]:
# # The following workaround is needed to run the jupyter notebook in docker container.
import os
import subprocess

# Run the script in a shell, capture its environment
PDK = "ihp-sg13g2"
command = f"bash -c 'source /foss/tools/sak/iic-pdk-script.sh {PDK} && source ~/.bashrc && env'"
result = subprocess.run(command, capture_output=True, text=True, shell=True)

# Parse environment variables from the output
for line in result.stdout.splitlines():
    key, _, value = line.partition("=")
    if key and value:
        os.environ[key] = value
        
os.environ["PATH"] += ":/foss/tools/bin"

# Now they are in your current Python process
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("SPICE_USERINIT_DIR:", os.environ.get("SPICE_USERINIT_DIR"))

# Test ngspice
!ngspice -v

## Library Imports

In [ ]:
import logging

from pathlib import Path

from symxplorer.spice_engine                import Spicelib_Wrapper, Sim_Execution_Type
# from symxplorer.designer_tools              import Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction, Project_Setup
from symxplorer.designer_tools              import Project_Setup
from symxplorer.optimization.nevergrad      import Nevergrad_Spice_Constraint_Satisfaction
from symxplorer.logging                     import setup_loggers

logger = logging.getLogger("SymXplorer.jupyter")
logger.info("Spicelib_Wrapper imported successfully.")

# Instantiations


## Loading the project config

In [ ]:
# ----------------------------
# Instantiations
# ----------------------------
project_setup_yaml = Path(f"/foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/spice/project_setup.yaml")
_ = setup_loggers()

# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

## Create a SPICE simulator wrapper

In [ ]:
# (2) Create the Spice Simulator Wrapper
netlist_filename = Path(PROJECT_SETUP.ws_root) / Path(PROJECT_SETUP.netlist)
output_folder    = Path(PROJECT_SETUP.ws_root) / Path(PROJECT_SETUP.outdir)

wrapper = Spicelib_Wrapper(
    project_name=PROJECT_SETUP.name,
    netlist_filename=netlist_filename,
    output_folder=output_folder,
    sim_execution_t=Sim_Execution_Type.RUN_AND_WAIT,  # only RUN_AND_WAIT is supported as of now...,
    path_to_simulator=Path("/foss/tools/bin/ngspice"),
    verbose=False
    )
wrapper

## Create an optimizer object

In [ ]:
circuit_optimizer = Nevergrad_Spice_Constraint_Satisfaction(
    spicelib_wrapper=wrapper,
    setup_obj=PROJECT_SETUP
)
circuit_optimizer

## Sanity Check

In [ ]:
wrapper.run_sanity_check(
    use_editor=True,
    sim_execution_t=Sim_Execution_Type.RUN_NOW
)

# Main Body

## Optimization

In [ ]:
circuit_optimizer.parameterize()

In [ ]:
_ = circuit_optimizer.optimize()

In [ ]:
circuit_optimizer.plot_score(save_path=project_setup_yaml.parent / "loss_curve.html", show=True)

## Inspection & Visualization

### (1) Best Param

In [ ]:
out = circuit_optimizer.get_best_params()
if out is not None: 
    best_param, loss, metadata = out
# metadata

In [ ]:
# Print parameter sizes (convert to u)
for param in best_param:
    print(f"{param}: {best_param[param] :0.2e}")

In [ ]:
circuit_optimizer.plot_solution(best_param, show_plot=True, trace_name="out")

### (3) Metric Trace

In [ ]:
_ = circuit_optimizer.plot_optimization_trace(metric_x='pm', metric_y='ugf', show=True)

In [ ]:
circuit_optimizer.plot_score_value_by_spec(spec_name="dcgain", show=True)
circuit_optimizer.plot_score_value_by_spec(spec_name="ugf", show=True)
circuit_optimizer.plot_score_value_by_spec(spec_name="pm", show=True)

### (4) Design Space Exploration

In [ ]:
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_pfet_load_w", param_y="x_dut_pfet_load_l", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_nfet_input_w", param_y="x_dut_nfet_input_l", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_nfet_mirror_w", param_y="x_dut_nfet_mirror_l", show=True)

# Checkpointing

In [ ]:
name = str(PROJECT_SETUP.ws_root/Path("sizing/data/checkpoint"))
circuit_optimizer.save_checkpoint(name=name)